In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [10]:
%pip install --upgrade --quiet yfinance

Note: you may need to restart the kernel to use updated packages.


In [11]:
from langgraph.graph import MessagesState
from langgraph.types import Command
from typing import Literal
from langchain_core.messages import HumanMessage

from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

from langchain.agents import create_agent

market_research_agent = create_agent(
    llm,
    tools=[YahooFinanceNewsTool()],
    system_prompt='You are a market researcher. Provide fact only not opinions'
)

def market_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = market_research_agent.invoke(state)
    print(f'Market research agent result: {result}')
    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='market_research')]},
        goto='supervisor'
    )

In [13]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [18]:
import yfinance as yf

from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> dict:
    """Given a stock ticker, return the price data for the past month"""
    stock_info = yf.download(ticker, period='1mo').to_dict()
    return stock_info